<a href="https://www.kaggle.com/code/tanimationtv/starter-notebook-for-name-that-llm-competition?scriptVersionId=366926658" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Name That LLM: Starter Baseline (TF-IDF + Logistic Regression)


In [1]:
import pathlib

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline

CANDIDATES = [
    pathlib.Path('/kaggle/input/competitions/name-that-llm'),
    pathlib.Path('/kaggle/input/name-that-llm'),
]
DATA_DIR = next(p for p in CANDIDATES if (p / 'train.csv').exists())

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train shape: {train_df.shape} | Unique models: {train_df["model"].nunique()} | Prompts: {train_df["prompt_id"].nunique()}')
print(f'Test shape: {test_df.shape}')
train_df.head(3)

Train shape: (2699, 4) | Unique models: 18 | Prompts: 150
Test shape: (3, 2)


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash


In [2]:
# Build Word + Character N-Gram TF-IDF + Logistic Regression Pipeline
model_pipeline = Pipeline(
    [
        (
            'features',
            FeatureUnion(
                [
                    (
                        'word_tfidf',
                        TfidfVectorizer(
                            ngram_range=(1, 2),
                            max_features=25000,
                            sublinear_tf=True,
                        ),
                    ),
                    (
                        'char_tfidf',
                        TfidfVectorizer(
                            analyzer='char',
                            ngram_range=(3, 5),
                            max_features=35000,
                            sublinear_tf=True,
                        ),
                    ),
                ]
            ),
        ),
        ('clf', LogisticRegression(C=2.0, max_iter=1000)),
    ]
)

# 5-fold CV that keeps each prompt in a single fold (test prompts are unseen), scored like the leaderboard
cv_scores = cross_val_score(
    model_pipeline,
    train_df['text'].fillna(''),
    train_df['model'],
    groups=train_df['prompt_id'],
    cv=GroupKFold(n_splits=5),
    scoring='balanced_accuracy',
    n_jobs=-1,
)
print(f'5-Fold Grouped CV Balanced Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# Fit on full training set
model_pipeline.fit(train_df['text'].fillna(''), train_df['model'])

5-Fold Grouped CV Balanced Accuracy: 0.3167 +/- 0.0123


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_features=25000,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 sublinear_tf=True)),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_features=35000,
                                                                 ngram_range=(3,
                                                                              5),
                                                                 sublinear_tf=True))])),
                ('clf', LogisticRegression(C=2.0, max_iter=1000))])

In [3]:
# Batch inference on test.csv (1 example row interactively; the full hidden test set during the Leaderboard Rerun)
test_preds = model_pipeline.predict(test_df['text'].fillna(''))

submission = pd.DataFrame(
    {
        'id': test_df['id'],
        'model': test_preds,
    }
)

submission.to_parquet('submission.parquet', index=False)
print(f'Saved submission.parquet with {len(submission)} rows.')
submission.head(10)

Saved submission.parquet with 3 rows.


,id,model
0,00353fb1c541,gemini-3.6-flash
1,005f5a9a6139,gpt-5.6-terra
2,0062a53d5487,gemini-3.7-flash


In [4]:
"""
FAST FINAL: DeBERTa-Large embeddings + SVM + Ridge + style LR + MLP + LR-on-embeddings, equal-weight blend.
- External data capped at 400 passages per model (keeps embedding time bounded).
- Embeddings cached to /kaggle/working/emb_cache.npz (restart or resubmit does not recompute).
- 3 layers (3072 dims), max_len 256.
- Validation: 70/30 split by prompt. Final test scores: refit on train + validation + external.
Paste as one cell. GPU on for the embedding step (cached afterwards).
"""
import glob, os, re, time, warnings, pathlib
import numpy as np, pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
T0 = time.time()
def log(*a): print(f"[{time.time() - T0:7.1f}s]", *a, flush=True)
def ba(t, s): return balanced_accuracy_score(t, np.asarray(s).argmax(1))

USE_STANDIN = os.environ.get("USE_STANDIN") == "1"     # testing only: random vectors instead of DeBERTa
REFIT = os.environ.get("REFIT", "1") == "1"            # refit on all labelled data for the test predictions
CAP_PER_MODEL = 400
EMB_CACHE = "/kaggle/working/emb_cache.npz"
os.makedirs("/kaggle/working", exist_ok=True)

# ============================================================================
# 1. DATA
# ============================================================================
def find_data_dir():
    if os.environ.get("DATA_DIR"): return pathlib.Path(os.environ["DATA_DIR"])
    for pat in ("/kaggle/input/**/train.csv", "/kaggle/input/*/train.csv"):
        for h in sorted({os.path.dirname(p) for p in glob.glob(pat, recursive=True)}):
            if os.path.exists(os.path.join(h, "test.csv")): return pathlib.Path(h)
    raise FileNotFoundError("train.csv not found")

DATA_DIR = find_data_dir()
train_full = pd.read_csv(DATA_DIR / "train.csv"); test_df = pd.read_csv(DATA_DIR / "test.csv")
for df in (train_full, test_df): df["text"] = df["text"].fillna("").astype(str)
CLASSES = np.array(sorted(train_full["model"].unique())); NC = len(CLASSES)

# --- external passages (same filters as before), capped per model ---
MIN_W, MAX_W, MIN_CHARS = 35, 280, 200
def load_external():
    out = []
    for p in glob.glob("/kaggle/input/**/passages.csv", recursive=True):
        if "/name-that-llm/" in p: continue
        try:
            d = pd.read_csv(p)
            if "passes_funnel" in d.columns: d = d[d["passes_funnel"].astype(bool)]
            if "is_refusal" in d.columns: d = d[~d["is_refusal"].astype(bool)]
            if {"text", "model"} <= set(d.columns):
                d = d[["text", "model"]].dropna(subset=["model"])
                d = d[d["model"].isin(CLASSES)].copy(); d["text"] = d["text"].fillna("").astype(str)
                nw = d["text"].str.split().str.len()
                out.append(d[nw.between(MIN_W, MAX_W) & (d["text"].str.len() > MIN_CHARS)])
        except Exception: pass
    if not out: return pd.DataFrame(columns=["text", "model"])
    ext = pd.concat(out, ignore_index=True).drop_duplicates(subset=["text"])
    return (ext.groupby("model", group_keys=False)
               .apply(lambda d: d.sample(min(len(d), CAP_PER_MODEL), random_state=42))
               .reset_index(drop=True))

EXT = load_external()
E_X = EXT["text"].values if len(EXT) else np.array([], dtype=object)
E_Y = np.searchsorted(CLASSES, EXT["model"].values) if len(EXT) else np.array([], dtype=int)
log(f"external rows used: {len(E_X)}")

# --- 70/30 split by prompt (validation = prompts the models never saw) ---
tr_idx, va_idx = next(GroupShuffleSplit(1, train_size=0.7, random_state=42).split(
    train_full, groups=train_full["prompt_id"].astype(str)))
train_df, val_df = train_full.iloc[tr_idx], train_full.iloc[va_idx]
X_train, y_train = train_df["text"].values, np.searchsorted(CLASSES, train_df["model"].values)
X_val, y_val = val_df["text"].values, np.searchsorted(CLASSES, val_df["model"].values)
XT = test_df["text"].values
X_all = np.concatenate([X_train, X_val]); y_all = np.concatenate([y_train, y_val])
log(f"train {len(X_train)} | val {len(X_val)} | test {len(XT)}")

# ============================================================================
# 2. EMBEDDINGS (DeBERTa-Large, 3 layers, cached)
# ============================================================================
def find_large_model():
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        if "deberta" in cfg.lower() and "small" not in cfg.lower(): return os.path.dirname(cfg)
    return None

ALL_TEXTS = {"train": X_train, "val": X_val, "test": XT, "ext": E_X}
if USE_STANDIN:
    rng = np.random.default_rng(0)
    EMB = {k: rng.normal(size=(len(v), 3072)).astype(np.float32) for k, v in ALL_TEXTS.items()}
elif os.path.exists(EMB_CACHE):
    z = np.load(EMB_CACHE); EMB = {k: z[k] for k in ALL_TEXTS}
    log("embeddings loaded from cache")
else:
    import torch
    from transformers import AutoModel, AutoTokenizer
    MODEL_PATH = find_large_model()
    if not MODEL_PATH: raise FileNotFoundError("Attach microsoft/deberta-v3-large via Add Input -> Models.")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu": log("WARNING: no GPU on, this will be slow. Turn the GPU on for this step.")
    tok = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModel.from_pretrained(MODEL_PATH).eval().to(device)
    if device == "cuda": model = model.half()
    n_layers = model.config.num_hidden_layers
    LAYERS = sorted({n_layers // 2, (3 * n_layers) // 4, n_layers})

    @torch.inference_mode()
    def embed(texts, batch=16, max_len=256):
        chunks = []
        for i in range(0, len(texts), batch):
            enc = tok(list(texts[i:i + batch]), truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(device)
            hs = model(**enc, output_hidden_states=True).hidden_states
            mf = enc["attention_mask"].unsqueeze(-1).float()
            chunks.append(np.hstack([((hs[L].float() * mf).sum(1) / mf.sum(1).clamp(min=1)).cpu().numpy() for L in LAYERS]))
        return np.vstack(chunks).astype(np.float32)

    EMB = {k: (embed(v) if len(v) else np.zeros((0, model.config.hidden_size * len(LAYERS)), np.float32))
           for k, v in ALL_TEXTS.items()}
    np.savez(EMB_CACHE, **EMB)
    log(f"embeddings cached to {EMB_CACHE}")
    del model; torch.cuda.empty_cache()
log(f"embedding dims: {EMB['train'].shape[1]}")

# ============================================================================
# 3. MODELS (fit once, predict any set of targets)
# ============================================================================
def align(A, cls):
    A = np.asarray(A, dtype=np.float64)
    if A.shape[1] == NC: return A
    out = np.full((A.shape[0], NC), A.min() - 1.0); out[:, cls] = A; return out

def rownorm(s):
    s = np.asarray(s, dtype=np.float64)
    return (s - s.mean(1, keepdims=True)) / (s.std(1, keepdims=True) + 1e-9)

def style_feats(texts):
    rows = []
    for t in texts:
        nw, nc = max(len(t.split()), 1), max(len(t), 1)
        alpha = max(sum(c.isalpha() for c in t), 1)
        rows.append([np.log1p(nc), np.log1p(nw), nc / nw,
                     (t.count("—") + t.count("–")) / nc * 1000, t.count(";") / nc * 1000,
                     t.count(",") / nc * 1000, len(re.findall(r"\*\*.+?\*\*", t)),
                     len(re.findall(r"^\s*[\*\-•]\s+", t, re.M)),
                     sum(c.isupper() for c in t) / alpha, len(re.findall(r"[​-‍﻿]", t))])
    return np.nan_to_num(np.array(rows, dtype=np.float64))

def run_models(fit_texts, fit_emb, fit_sty, fit_y, targets):
    """targets: list of (texts, emb, style). Returns one dict of block scores per target."""
    cv = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), lowercase=False, sublinear_tf=True, min_df=2, max_features=200000)
    wv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=False, sublinear_tf=True, min_df=2, max_features=100000)
    A_fit = hstack([cv.fit_transform(fit_texts), wv.fit_transform(fit_texts)]).tocsr()
    sc_e = RobustScaler().fit(fit_emb); sc_s = RobustScaler().fit(fit_sty)
    E_fit, S_fit = sc_e.transform(fit_emb), sc_s.transform(fit_sty)

    svm = LinearSVC(C=0.5, class_weight="balanced", dual=False, max_iter=5000).fit(A_fit, fit_y)
    ridge = OneVsRestClassifier(RidgeClassifier(alpha=1.0, class_weight="balanced")).fit(A_fit, fit_y)
    sty = OneVsRestClassifier(LogisticRegression(C=1.0, class_weight="balanced", solver="liblinear", max_iter=2000)).fit(S_fit, fit_y)
    mlp = MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.01, max_iter=300, random_state=42, early_stopping=True).fit(E_fit, fit_y)
    lr_e = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000).fit(E_fit, fit_y)
    log("  base models fitted")

    outs = []
    for texts, emb, sty_ in targets:
        A = hstack([cv.transform(texts), wv.transform(texts)]).tocsr()
        E, S = sc_e.transform(emb), sc_s.transform(sty_)
        outs.append({
            "svm": align(svm.decision_function(A), svm.classes_),
            "ridge": align(ridge.decision_function(A), ridge.classes_),
            "style": align(sty.decision_function(S), sty.classes_),
            "mlp": align(np.log(mlp.predict_proba(E) + 1e-9), mlp.classes_),
            "lr_emb": align(np.log(lr_e.predict_proba(E) + 1e-9), lr_e.classes_),
        })
    return outs

def blend(block_dict):
    return sum(rownorm(v) for v in block_dict.values()) / len(block_dict)

# ============================================================================
# 4. VALIDATION (fit on 70% + external, score the 30%)
# ============================================================================
fit_texts = np.concatenate([X_train, E_X]); fit_y = np.concatenate([y_train, E_Y])
fit_emb = np.vstack([EMB["train"], EMB["ext"]]) if len(E_X) else EMB["train"]
fit_sty = style_feats(fit_texts)
val_sty = style_feats(X_val)

val_blocks, = run_models(fit_texts, fit_emb, fit_sty, fit_y, [(X_val, EMB["val"], val_sty)])
for k, v in val_blocks.items(): log(f"  {k:7s} val {ba(y_val, v):.4f}")
val_score = ba(y_val, blend(val_blocks))
log(f"★ EQUAL-WEIGHT VALIDATION BALANCED ACCURACY: {val_score:.4f} ★")

# ============================================================================
# 5. FINAL TEST PREDICTIONS (refit on train + validation + external)
# ============================================================================
test_sty = style_feats(XT)
if REFIT:
    all_texts = np.concatenate([X_all, E_X]); all_y = np.concatenate([y_all, E_Y])
    all_emb = np.vstack([EMB["train"], EMB["val"], EMB["ext"]]) if len(E_X) else np.vstack([EMB["train"], EMB["val"]])
    all_sty = style_feats(all_texts)
    test_blocks, = run_models(all_texts, all_emb, all_sty, all_y, [(XT, EMB["test"], test_sty)])
    log("final models refit on all labelled data")
else:
    test_blocks, = run_models(fit_texts, fit_emb, fit_sty, fit_y, [(XT, EMB["test"], test_sty)])

pred = CLASSES[blend(test_blocks).argmax(1)]
sub = pd.DataFrame({"id": test_df["id"].values, "model": pred})
sub.to_parquet("/kaggle/working/submission.parquet", index=False)
log(f"✓ saved /kaggle/working/submission.parquet ({len(sub)} rows, {len(set(pred))} classes predicted)")

[    1.1s] external rows used: 7200
[    1.1s] train 1889 | val 810 | test 3


[transformers] The tokenizer you are loading from '/kaggle/input/datasets/prvsiyan/deberta-v3-large-weights/deberta-v3-large' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: /kaggle/input/datasets/prvsiyan/deberta-v3-large-weights/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identi

[  249.2s] embeddings cached to /kaggle/working/emb_cache.npz
[  249.3s] embedding dims: 3072
[  490.8s]   base models fitted
[  492.7s]   svm     val 0.3383
[  492.7s]   ridge   val 0.3383
[  492.7s]   style   val 0.1827
[  492.7s]   mlp     val 0.4210
[  492.7s]   lr_emb  val 0.4049
[  492.7s] ★ EQUAL-WEIGHT VALIDATION BALANCED ACCURACY: 0.4420 ★
[  759.7s]   base models fitted
[  759.9s] final models refit on all labelled data
[  759.9s] ✓ saved /kaggle/working/submission.parquet (3 rows, 3 classes predicted)
